# Lab 37: Classical Machine Learning: Classification and Regression

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Before deep learning, machine learning used simple models that learn from a table of numbers. They are fast, small, easy to explain, and often good enough. They run in milliseconds on the CPU of the Jetson. In this lab you train and compare four models on two kinds of problem: **classification** (which class? a type of wine) and **regression** (how much? a temperature). You always compare with a **baseline**, use **cross-validation** to be fair, and read what the models learned.

**By the end of this lab you can:**
1. Explain the difference between classification and regression, and name a model for each.
2. Train logistic regression, k-nearest neighbours, a decision tree and a random forest, and compare them with a baseline.
3. Evaluate classification with accuracy, F1 and a confusion matrix, and regression with MAE, RMSE and R-squared.
4. Use cross-validation to get a score that does not depend on one lucky split.
5. Read what a model learned (tree rules, feature importance, coefficients) and choose a model for a job.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Classification of this lab.** *Training from scratch* with scikit-learn. The datasets are small and come with the library, so nothing is downloaded.

## 2. Background

**Two kinds of problem.**

| | Classification | Regression |
|---|---|---|
| The answer is | A class (a name) | A number |
| Example | Which wine is it? Is the fan needed? | What will the temperature be? |
| Baseline | Always the most common class | Always the average |
| Metrics | Accuracy, precision, recall, F1, confusion matrix | MAE, RMSE, R-squared |

**The models.**

| Model | Idea | Good | Weak |
|---|---|---|---|
| **Logistic regression** (classification) / **linear regression** | Draws a straight line (or plane) between the classes, or through the numbers | Fast, explains itself (the coefficients) | Cannot learn curves |
| **k-nearest neighbours** | Looks at the `k` most similar training examples and takes a vote | Easy to understand | Slow with big data; needs scaled features |
| **Decision tree** | A set of yes/no questions (*is the alcohol above 12.8?*) | Easy to read; no scaling | Easy to overfit; unstable |
| **Random forest** | Many trees vote | Strong, few settings | Larger, harder to read |

**Metrics.**
- **F1** is a mix of precision and recall in one number (1 is perfect).
- **MAE** (mean absolute error): the average size of the mistake, in the unit of the answer (degrees).
- **RMSE**: like MAE, but big mistakes count more.
- **R-squared**: 1 is perfect, 0 is no better than the average, negative is worse than the average.

**Cross-validation.** One split can be lucky. In 5-fold cross-validation the data is cut into 5 parts. The model is trained 5 times, each time testing on another part. You report the **mean and the spread**. Use it to compare models; keep a final test set for the one honest exam.

**A baseline first.** A model is only good if it beats the dumb answer. Always print the baseline first.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(37)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (10 min)

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

SEED = 11
print("Ready.")

---
# Part B: Classification, Which Wine? (55 min)

### Step 1: The Data
The wine dataset has 178 wines from three growers. For each wine there are 13 measurements (alcohol, colour, ...). The label is the grower (class 0, 1 or 2). The cell splits the data 70 % / 30 % with the same share of each class in both parts.

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
print("Rows:", X.shape[0], "| features:", X.shape[1], "| classes:", list(wine.target_names))
print("Features:", list(wine.feature_names)[:6], "...")
print("Examples per class:", np.bincount(y))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
print("Train:", len(X_train), "| test:", len(X_test))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many examples are in each class:
- What is the accuracy of a model that always says the biggest class (calculate it):

### Step 2: The Baseline and Four Models
Three of the models need **scaled** features, so the cell puts a `StandardScaler` in front of them with `make_pipeline` (the scaler learns from the training data only). Each model is scored with 5-fold cross-validation **on the training set**, and then once on the test set.

In [ ]:
models = {
    "baseline (most common)": DummyClassifier(strategy="most_frequent"),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "k nearest neighbours (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "random forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=SEED),
}
rows = []
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    rows.append((name, cv.mean(), cv.std(), (predicted == y_test).mean(), f1_score(y_test, predicted, average="macro")))
table = pd.DataFrame(rows, columns=["model", "cv accuracy", "cv spread", "test accuracy", "test F1"]).set_index("model")
print(table.round(3))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The baseline accuracy and the best model:
- Which model has the biggest spread in the cross-validation, and what does it mean:
- Is the best model much better than the second best? Can you be sure from 54 test examples:

### Step 3: Where Does It Go Wrong?
The confusion matrix of the random forest on the test set. A perfect model has numbers only on the diagonal.

In [ ]:
forest = models["random forest (100 trees)"]
ConfusionMatrixDisplay(confusion_matrix(y_test, forest.predict(X_test)), display_labels=wine.target_names).plot(cmap="Blues")
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which two classes were mixed up most:
- How many test examples were wrong:

### Step 4: Read What the Models Learned
A decision tree can be printed as rules. A random forest tells how much each feature mattered. A logistic regression has one coefficient per feature and class. This is the **explainability** of classical models.

In [ ]:
tree = models["decision tree (depth 3)"]
print(export_text(tree, feature_names=list(wine.feature_names)))
importance = pd.Series(forest.feature_importances_, index=wine.feature_names).sort_values(ascending=False)
print("Most important features (random forest):")
print(importance.head(5).round(3))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The first question the tree asks, and why that feature:
- The three most important features of the forest:
- Which of the two explanations would you show to a customer, and why:

---
# Part C: Regression, How Warm Will It Be? (45 min)

### Step 1: The Data
A synthetic table: the temperature inside a room depends on the **outside temperature**, the **hour of the day** and whether the **window is open**, plus a little noise. You know the true rule, so you can check what the models found: inside = 21 + 0.5 x outside + 2 x sin(hour) - 3 x window + noise.

In [ ]:
rng = np.random.default_rng(SEED)
M = 500
outside = rng.uniform(20, 36, M)
hour = rng.uniform(0, 24, M)
window = rng.integers(0, 2, M)
inside = 21 + 0.5 * (outside - 28) + 2 * np.sin((hour - 9) / 24 * 2 * np.pi) - 3 * window + rng.normal(0, 0.6, M)
frame = pd.DataFrame({"outside": outside, "hour": hour, "window": window, "inside": inside})
print(frame.head(4).round(1))

Xr = frame[["outside", "hour", "window"]].to_numpy()
yr = frame["inside"].to_numpy()
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.30, random_state=SEED)

### Step 2: The Baseline and Four Models
The baseline answers the average temperature every time. The metrics are in degrees (MAE and RMSE) and R-squared.

In [ ]:
reg_models = {
    "baseline (average)": DummyRegressor(strategy="mean"),
    "linear regression": LinearRegression(),
    "ridge regression": Ridge(alpha=1.0),
    "decision tree (depth 4)": DecisionTreeRegressor(max_depth=4, random_state=SEED),
    "random forest (100 trees)": RandomForestRegressor(n_estimators=100, random_state=SEED),
}
rows = []
for name, model in reg_models.items():
    model.fit(Xr_train, yr_train)
    predicted = model.predict(Xr_test)
    rows.append((name, mean_absolute_error(yr_test, predicted), mean_squared_error(yr_test, predicted) ** 0.5, r2_score(yr_test, predicted)))
print(pd.DataFrame(rows, columns=["model", "MAE (C)", "RMSE (C)", "R-squared"]).set_index("model").round(3))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- MAE of the baseline and of the best model:
- Which model wins, and is it a surprise (the true rule is almost a straight line plus a sine):
- What does an R-squared of 0 mean for the baseline:

### Step 3: Predicted against Real
A good model puts the points on the diagonal line. The right picture shows the **residuals** (real minus predicted): they should be spread around 0 without a pattern.

In [ ]:
best = reg_models["random forest (100 trees)"]
predicted = best.predict(Xr_test)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].scatter(yr_test, predicted, s=10)
axes[0].plot([yr.min(), yr.max()], [yr.min(), yr.max()], "r--")
axes[0].set_xlabel("real (C)")
axes[0].set_ylabel("predicted (C)")
axes[1].hist(yr_test - predicted, bins=20)
axes[1].set_xlabel("real minus predicted (C)")
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Are the points close to the red line:
- Is the histogram centred on 0? What would it mean if it were not:

### Step 4: Read the Coefficients
Linear regression gives one number per feature: how much the answer changes when the feature goes up by one. Compare them with the true rule: `outside` should be about 0.5 and `window` about -3.

In [ ]:
linear = reg_models["linear regression"]
for name, value in zip(["outside", "hour", "window"], linear.coef_):
    print(f"{name:<8} coefficient {value:+.2f}")
print(f"intercept {linear.intercept_:+.2f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The coefficients of outside and window, and how close they are to 0.5 and -3:
- Why is the coefficient of hour not close to the true effect (the effect is a sine, not a straight line):

---
# Part D: Choose a Model (30 min)

### Step 1: Speed and Size
For an edge device the time to predict and the size also count. The cell measures how long each classifier needs to predict one wine (the mean of 200 predictions).

In [ ]:
one = X_test[:1]
for name, model in models.items():
    started = time.perf_counter()
    for _ in range(200):
        model.predict(one)
    print(f"{name:<30} {(time.perf_counter() - started) / 200 * 1000:7.2f} ms per prediction")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which model is the fastest and which is the slowest:
- Fill in your choice: for a battery sensor that must decide in 1 ms I choose ... because ...:
- For a medical decision that must be explained to a patient I choose ... because ...:

---
# Part E: Make It Your Own and Present (30 min)
Choose **one**:

- Change `k` of the nearest neighbours (1, 3, 11, 25) and write the cross-validation accuracy of each. Which `k` is best?
- Change the depth of the tree (1, 2, 3, 5, 10) and find where the test accuracy stops improving.
- Remove the `StandardScaler` from the k-nearest-neighbours pipeline. How much worse does it get, and why?
- Use your own CSV (Lab 36: `train_ready.csv`) with the models of Part B.

Then give a **3-minute demonstration**: show the baseline, the table of four models, the confusion matrix and the explanation of the best model.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw:
- Which model you would use for the wine problem and why:

> **Where next.** In Lab 38 you open the box of a neural network and build one from scratch with NumPy: the forward pass, the loss and the gradients.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The baseline and the four classification models are compared with cross-validation and a test set | |
| 2 | The confusion matrix is read | |
| 3 | The tree rules and the feature importance are printed and explained | |
| 4 | The four regression models are compared with MAE, RMSE and R-squared, with a baseline | |
| 5 | The coefficients are compared with the true rule | |
| 6 | The speed of the models is measured and a model is chosen for two different jobs | |
| 7 | One improvement is done and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is the difference between classification and regression? Give one example of each.:
- 2. Why do you compare a model with a baseline?:
- 3. What does 5-fold cross-validation do? Why report the spread?:
- 4. Why do k-nearest neighbours and logistic regression need scaled features, and a tree does not?:
- 5. What does a confusion matrix show that accuracy does not?:
- 6. Explain MAE and R-squared in one sentence each.:
- 7. Why is a decision tree easy to explain? Why can it overfit?:
- 8. How does a random forest differ from one tree?:
- 9. Your model has R-squared = 0.99 on the training set and 0.4 on the test set. What is wrong?:
- 10. Why are classical models a good choice for a small sensor device?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `ModuleNotFoundError: sklearn` | scikit-learn is not in the environment | Do Lab 19 (it installs scikit-learn) or ask your instructor |
| `ConvergenceWarning` from the logistic regression | The solver needs more steps | It is a warning, not an error. The cell already uses `max_iter=1000`; scale the features to remove it. |
| The scores are different from your classmate's | Another seed or another split | Run from the top; the seed is 11 |
| The k-nearest-neighbours score is low | The features were not scaled | Use the pipeline with `StandardScaler` |
| The pictures do not appear | The plot cell did not run | Run it again; the picture is under the cell |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(37)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- scikit-learn, Choosing the right estimator: https://scikit-learn.org/stable/machine_learning_map.html
- scikit-learn, Cross-validation: https://scikit-learn.org/stable/modules/cross_validation.html
- StatQuest (videos on decision trees and random forests): https://statquest.org/